# ReaVector AI — Kaggle main GPU (T4 ×2)

Personal / owner use only. Pulls one queued job from the Render bridge, runs it on a pinned T4, uploads the GLB back. No public serving.

**GPU 0** = images / LoRA / textures · **GPU 1** = video / 3D.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/rizbiehelp-sys/reavectors.git
%cd reavectors
!pip -q install numpy pillow trimesh requests rembg onnxruntime

## Bridge settings
Paste your `AI_BRIDGE_KEY` (from Render → Environment). Keep it in this notebook only — never commit it.

In [ ]:
import os, json, requests
BRIDGE = os.environ.get('BRIDGE_URL', 'https://reavectors.onrender.com')
KEY = os.environ.get('BRIDGE_KEY', 'PASTE_AI_BRIDGE_KEY_HERE')
H = {'X-Bridge-Key': KEY}
def pending():
    r = requests.get(f'{BRIDGE}/api/ai/bridge/jobs', params={'status':'pending'}, headers=H, timeout=30)
    r.raise_for_status(); return r.json()['jobs']
def claim(jid):
    return requests.post(f'{BRIDGE}/api/ai/bridge/jobs/{jid}/claim', headers=H, timeout=30).json()
print('bridge =', BRIDGE, '| key set =', KEY != 'PASTE_AI_BRIDGE_KEY_HERE')
pending()

## Install the 3D model (TripoSR by default)
For SF3D swap the clone/requirements. Hunyuan3D-2.1 (~24 GB) will not fit a T4.

In [ ]:
!git clone https://github.com/VAST-AI-Research/TripoSR /opt/TripoSR
!pip -q install -r /opt/TripoSR/requirements.txt
os.environ['PYTHONPATH'] = '/opt/TripoSR:' + os.environ.get('PYTHONPATH','')

## Run the next queued job
Downloads the job's input image, runs on GPU 1 (3D), uploads the GLB. The website can then load the artifact URL.

In [ ]:
import subprocess, tempfile
jobs = pending()
if not jobs:
    print('no pending jobs')
else:
    j = claim(jobs[0]['id'])
    print('claimed', j)
    img = os.path.join('/kaggle/working', os.path.basename(j.get('input_url') or 'input.png'))
    with open(img, 'wb') as f:
        f.write(requests.get(j['input_url'], timeout=60).content)
    out = '/kaggle/working/result.glb'
    gpu = '1' if j['model'] in ('triposr','sf3d','hunyuan3d') else '0'
    r = subprocess.run(['python','scripts/gpu_job.py','--gpu',gpu,'--model',j['model'],'--image',img,'--out',out], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if os.path.exists(out):
        up = requests.post(f"{BRIDGE}/api/ai/bridge/jobs/{j['id']}/result", headers=H,
                           files={'file': open(out,'rb')}, data={'filename':'result.glb'}, timeout=120)
        print(up.status_code, up.json())
    else:
        requests.post(f"{BRIDGE}/api/ai/bridge/jobs/{j['id']}/fail", headers=H, json={'error': r.stderr[-500:]})

## Benchmark each T4 separately
Writes a report (seconds, peak VRAM, failures). Run once per GPU.

In [ ]:
!python scripts/ai_bench.py --gpu 1 --models triposr,sf3d --image /kaggle/working/result.glb 2>/dev/null || true
!python scripts/ai_bench.py --list